# Overview

The objective of this capstone is to analyze reviews, extract insights, and understand sentiment.

IMDB receives thousands of customer reviews and feedback. However, manually analyzing this data is inefficient. The goal of this capstone is to develop an NLP-powered sentiment analysis system that automatically classifies reviews as positive or negative.

The labeled data set consists of 50,000 IMDB movie reviews, specially selected for sentiment analysis. The sentiment of reviews is binary, meaning the IMDB rating < 5 results in a sentiment score of 0, and rating >=7 have a sentiment score of 1. No individual movie has more than 30 reviews. The 25,000 review labeled training set does not include any of the same movies as the 25,000 review test set.

Input dataset: [Dataset](https://gperdrizet.github.io/FSA_devops/datasets/)

## Steps to Perform

### Task 1: Data Exploration & Preprocessing

- Explore customer reviews from the given dataset
- Perform text cleaning (lowercasing, punctuation, stop-words, lemmatization, etc..)
- Tokenize and vectorize the text if required

### Task 2: Sentiment analysis

- Build a sentiment classification model (positive:1 vs negative:0) using:
    - Traditional ML model using [NaiveBayes](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html) ( A good example was in the  lesson 38 Demo )
    - A custom Deep Learning model including LSTM layers
- Evaluate models using accuracy, F1-score, and confusion matrix
- Compare your NB, Custom LSTM, and your choice of a pre-trained BERT model from HuggingFace using the [transformers](https://pypi.org/project/transformers/) package.
- Optionally, you could train a transformer model yourself. This will take a lot of time / resources, but could be excellent practice and would increase your understanding of transformer architecture.

### Task 3: Predictions

- Get predicted Sentiments for the unlabeledTestData

In [14]:
# Install necessary libraries
%pip install pandas numpy transformers datasets scikit-learn optuna tensorflow keras optuna-integration[tfkeras] nltk --quiet

Note: you may need to restart the kernel to use updated packages.


In [15]:
# Import the libs and modules
import os
import nltk
import pandas as pd
import numpy as np
import tensorflow as tf
import keras as ks
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
import optuna
from optuna.integration import TFKerasPruningCallback
from nltk.stem import PorterStemmer
from nltk import WordNetLemmatizer
from transformers import BertTokenizer

nltk.download('wordnet')


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Rick\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [16]:
# pull the data into a dataframe for processing from a tsv file
df = pd.read_csv('labeledTrainData.tsv', sep='\t')
# Quick check on how the data looks
print(df.head())
print(df.info())
print(df.describe())
print(df['sentiment'].value_counts())

       id  sentiment                                             review
0  5814_8          1  With all this stuff going down at the moment w...
1  2381_9          1  \The Classic War of the Worlds\" by Timothy Hi...
2  7759_3          0  The film starts with a manager (Nicholas Bell)...
3  3630_4          0  It must be assumed that those who praised this...
4  9495_8          1  Superbly trashy and wondrously unpretentious 8...
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         25000 non-null  object
 1   sentiment  25000 non-null  int64 
 2   review     25000 non-null  object
dtypes: int64(1), object(2)
memory usage: 586.1+ KB
None
         sentiment
count  25000.00000
mean       0.50000
std        0.50001
min        0.00000
25%        0.00000
50%        0.50000
75%        1.00000
max        1.00000
sentiment
1    12500
0    12500
Name:

In [31]:
# If we have previously processed the data and created a file for the processed data, we can load that file instead
if os.path.exists('processed_labeledTrainData.tsv'):
    df = pd.read_csv('processed_labeledTrainData.tsv', sep='\t')
else:
    # Process the text into a usuable format for training the model
    # lower case all text
    df['review'] = df['review'].str.lower()
    # remove punctuation (REGEX pattern is Not Words and Not Spaces)
    df['review'] = df['review'].str.replace(r'[^\w\s]', '', regex=True)
    # use the sklearn stop word list and remove stop words
    df['review'] = df['review'].apply(lambda x: ' '.join([word for word in x.split() if word not in ENGLISH_STOP_WORDS]))
    # stem and lemmatize the text using nltk's PorterStemmer and WordNetLemmatizer
    df['review'] = df['review'].apply(lambda x: ' '.join([PorterStemmer().stem(word) for word in x.split()]))
    df['review'] = df['review'].apply(lambda x: ' '.join([WordNetLemmatizer().lemmatize(word) for word in x.split()]))
    # Check some of the reviews after processing
    print(df['review'].head())

    # Save the processed data to a new tsv file for future use
    df.to_csv('processed_labeledTrainData.tsv', sep='\t', index=False)

In [33]:
# Do the same for the unlabeled data if we have it and want to use it for testing the model
if os.path.exists('processed_unlabeledTestData.tsv'):
    df_unlabeled = pd.read_csv('processed_unlabeledTestData.tsv', sep='\t')
else:
    df_unlabeled = pd.read_csv('unlabeledTestData.tsv', sep='\t')
    df_unlabeled['review'] = df_unlabeled['review'].str.lower()
    df_unlabeled['review'] = df_unlabeled['review'].str.replace(r'[^\w\s]', '', regex=True)
    df_unlabeled['review'] = df_unlabeled['review'].apply(lambda x: ' '.join([word for word in x.split() if word not in ENGLISH_STOP_WORDS]))
    df_unlabeled['review'] = df_unlabeled['review'].apply(lambda x: ' '.join([PorterStemmer().stem(word) for word in x.split()]))
    df_unlabeled['review'] = df_unlabeled['review'].apply(lambda x: ' '.join([WordNetLemmatizer().lemmatize(word) for word in x.split()]))
    print(df_unlabeled['review'].head())
    df_unlabeled.to_csv('processed_unlabeledTestData.tsv', sep='\t', index=False)

0    natur film who main theme mortal nostalgia los...
1    movi disast disast film great action scene mea...
2    movi kid saw tonight child love point kid exci...
3    afraid dark left impress differ screenplay wri...
4    accur depict small time mob life film new jers...
Name: review, dtype: object


In [34]:
# Tokenize the text using HuggingFace's BertTokenizer into a new dataframe to keep variables separate for the different models
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
df_tokenized = df.copy()
# split reviews into the maximium sequence length of the BERT tokenizer (512 tokens) and tokenize the text
df_tokenized['review'] = df_tokenized['review'].apply(lambda x: tokenizer.encode(x, add_special_tokens=True, max_length=512, truncation=True))
# Check some of the tokenized reviews
print(df_tokenized['review'].head())
# print the amount of unique tokens in the tokenized reviews
unique_tokens = set()
for review in df_tokenized['review']:
    unique_tokens.update(review)
print(f'Unique tokens in the tokenized reviews: {len(unique_tokens)}')
# Check some of the tokenized reviews
print("Tokenized reviews:")
print(df_tokenized['review'].head())

0    [101, 4933, 2175, 2617, 1049, 3501, 4921, 2063...
1    [101, 4438, 2162, 2088, 5199, 14573, 2072, 763...
2    [101, 2143, 2707, 24951, 2290, 27969, 14854, 2...
3    [101, 4632, 2819, 10975, 15593, 2143, 4602, 21...
4    [101, 21688, 2140, 11669, 2072, 2180, 22196, 2...
Name: review, dtype: object
Unique tokens in the tokenized reviews: 18719
Tokenized reviews:
0    [101, 4933, 2175, 2617, 1049, 3501, 4921, 2063...
1    [101, 4438, 2162, 2088, 5199, 14573, 2072, 763...
2    [101, 2143, 2707, 24951, 2290, 27969, 14854, 2...
3    [101, 4632, 2819, 10975, 15593, 2143, 4602, 21...
4    [101, 21688, 2140, 11669, 2072, 2180, 22196, 2...
Name: review, dtype: object


In [36]:
# Do the same tokenization steps for the unlabeled data to keep in line
df_unlabeled_tokenized = df_unlabeled.copy()
df_unlabeled_tokenized['review'] = df_unlabeled_tokenized['review'].apply(lambda x: tokenizer.encode(x, add_special_tokens=True, max_length=512, truncation=True))
print("Tokenized unlabeled reviews:")
print(df_unlabeled_tokenized['review'].head())

Tokenized unlabeled reviews:
0    [101, 14085, 3126, 2143, 2040, 2364, 4323, 980...
1    [101, 9587, 5737, 4487, 20939, 2102, 4487, 209...
2    [101, 9587, 5737, 4845, 2387, 3892, 2775, 2293...
3    [101, 4452, 2601, 2187, 17894, 11234, 9000, 25...
4    [101, 16222, 3126, 17120, 2235, 2051, 11240, 2...
Name: review, dtype: object


In [41]:
# Vectorize all the tokens between labeled and unlabeled data to keep the same vectorization for both datasets
vectorizer = TfidfVectorizer(max_features=20000)
all_reviews = pd.concat([df_tokenized['review'], df_unlabeled_tokenized['review']])
vectorizer.fit(all_reviews)
X_train = vectorizer.transform(df_tokenized['review'])
X_test = vectorizer.transform(df_unlabeled_tokenized['review'])
y_train = df_tokenized['sentiment']
y_test = np.zeros(len(df_unlabeled_tokenized))  # Unlabeled data has no labels, so we can use a placeholder
print(f'Shape of X_train: {X_train.shape}')
print(f'Shape of X_test: {X_test.shape}')
print(f'Shape of y_train: {y_train.shape}')
print(f'Shape of y_test: {y_test.shape}')


AttributeError: 'list' object has no attribute 'lower'

In [ ]:
# Create a Naive Bayes model using the vectorized reviews and the sentiment labels
nb_model = MultinomialNB()
nb_model.fit(df_vectorized, df['sentiment'])
# Make predictions and evaluate the model
nb_predictions = nb_model.predict(df_vectorized)
print("Classification Report for Naive Bayes Model:")
print(classification_report(df['sentiment'], nb_predictions))
# Evaluate the model on the unlabeled data (we won't have true labels for this, so we'll just look at the distribution of predicted labels)
unlabeled_predictions = nb_model.predict(df_unlabeled_vectorized)
print("Predicted label distribution for unlabeled data:")
print(pd.Series(unlabeled_predictions).value_counts())

Classification Report for Naive Bayes Model:
              precision    recall  f1-score   support

           0       0.87      0.88      0.88     12500
           1       0.88      0.87      0.88     12500

    accuracy                           0.88     25000
   macro avg       0.88      0.88      0.88     25000
weighted avg       0.88      0.88      0.88     25000



ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- 10008
- 10009
- 10034
- 10044
- 10080
- ...
Feature names seen at fit time, yet now missing:
- 10004
- 10011
- 10039
- 10066
- 10067
- ...
